In [1]:
!python -m spacy download en_core_web_lg
!python -c "import nltk; nltk.download('punkt'); nltk.download('averaged_perceptron_tagger'); nltk.download('maxent_ne_chunker'); nltk.download('words'); nltk.download('punkt_tab'); nltk.download('maxent_ne_chunker_tab'); nltk.download('averaged_perceptron_tagger_eng')"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.4 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /usr/share/nltk_data...
[nltk_data]   Package words is already up-t

In [2]:
"""
pdf_text_extractor.py

Turns a PyMuPDF page into:
  1) a single plain-text string (what we feed to GLiNER), and
  2) a parallel array that maps every character in that string back to its
     exact bounding box + line id + font size on the page.

This is what lets us go from "GLiNER found the substring at [start:end)"
straight to "here are the rectangles on the page to redact".

Uses page.get_text("rawdict") because it gives per-character bboxes, which
is the only granularity fine enough to slice out a PII span (e.g. just the
"Sarthak Malvadkar" run inside a longer line of text) without dragging in
neighboring words.
"""

!pip install pymupdf
!pip install tools

from dataclasses import dataclass
from typing import List, Optional, Tuple

import pymupdf  # PyMuPDF


@dataclass
class CharInfo:
    ch: str
    bbox: Tuple[float, float, float, float]
    line_id: int
    font_size: float
    font: str


def extract_page_chars(page: "pymupdf.Page") -> Tuple[str, List[Optional[CharInfo]]]:
    """
    Returns (full_text, char_infos) where char_infos[i] describes
    full_text[i]. A line-break inserted between PDF "lines" has
    char_infos[i] == None (it has no bbox of its own).

    full_text[i] == char_infos[i].ch for every non-None entry, by
    construction -- this invariant is what makes offset -> bbox mapping
    reliable later on.
    """
    raw = page.get_text("rawdict")

    text_chars: List[str] = []
    char_infos: List[Optional[CharInfo]] = []
    line_id = 0

    for block in raw.get("blocks", []):
        if block.get("type") != 0:  # 0 = text block, 1 = image block
            continue
        for line in block.get("lines", []):
            wrote_any = False
            for span in line.get("spans", []):
                size = span.get("size", 10.0)
                font = span.get("font", "helv")
                for ch in span.get("chars", []):
                    text_chars.append(ch["c"])
                    char_infos.append(
                        CharInfo(
                            ch=ch["c"],
                            bbox=tuple(ch["bbox"]),
                            line_id=line_id,
                            font_size=size,
                            font=font,
                        )
                    )
                    wrote_any = True
            if wrote_any:
                # Separator between PDF lines. Gives GLiNER a natural
                # boundary and stops entities from being glued across
                # unrelated lines (e.g. a name at end of one line and an
                # address on the next).
                text_chars.append("\n")
                char_infos.append(None)
                line_id += 1

    return "".join(text_chars), char_infos


def entity_to_line_rects(start: int, end: int, char_infos: List[Optional[CharInfo]]):
    """
    Given a [start, end) character span in the text produced by
    extract_page_chars, return one rect per PDF line the span touches:

        [{"rect": pymupdf.Rect(...), "font_size": float, "font": str}, ...]

    Multi-line entities (e.g. a wrapped street address) come back as
    multiple rects, one per line, in reading order.
    """
    if start < 0 or end > len(char_infos) or start >= end:
        return []

    by_line = {}
    order = []
    for i in range(start, end):
        info = char_infos[i]
        if info is None:
            continue
        if info.line_id not in by_line:
            by_line[info.line_id] = []
            order.append(info.line_id)
        by_line[info.line_id].append(info)

    rects = []
    for lid in order:
        infos = by_line[lid]
        x0 = min(c.bbox[0] for c in infos)
        y0 = min(c.bbox[1] for c in infos)
        x1 = max(c.bbox[2] for c in infos)
        y1 = max(c.bbox[3] for c in infos)
        avg_size = sum(c.font_size for c in infos) / len(infos)
        rects.append(
            {
                "rect": pymupdf.Rect(x0, y0, x1, y1),
                "font_size": avg_size,
                "font": infos[0].font,
            }
        )
    return rects

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 70.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00


In [3]:
"""
entity_detector.py

Wraps a GLiNER-style model (anything exposing .predict_entities(text,
labels, threshold=...) -> List[{"start","end","text","label","score"}])
and adds two things GLiNER doesn't do for you:

  1. Chunking: gretel-gliner-bi-base-v1.0, like most GLiNER checkpoints,
     has a limited effective context window. A dense prospectus page can
     run several thousand characters, so we slide a window over the page
     text and offset the results back into page-absolute coordinates.

  2. Overlap resolution: with 40+ labels turned on at once (first_name,
     last_name, name, ...) it's common to get overlapping spans on the
     same text. We greedily keep the highest-confidence, longest span and
     drop anything that overlaps it.
"""

from typing import Any, Dict, List, Protocol
# import secondary_detectors


ALL_LABELS = [
    "medical_record_number",
    "date_of_birth",
    "ssn",
    "date",
    "first_name",
    "email",
    "last_name",
    "customer_id",
    "employee_id",
    "name",
    "street_address",
    "phone_number",
    "ipv4",
    "credit_card_number",
    "license_plate",
    "address",
    "user_name",
    "device_identifier",
    "bank_routing_number",
    "date_time",
    "company_name",
    "unique_identifier",
    "biometric_identifier",
    "account_number",
    "city",
    "certificate_license_number",
    "time",
    "postcode",
    "vehicle_identifier",
    "coordinate",
    "country",
    "api_key",
    "ipv6",
    "password",
    "health_plan_beneficiary_number",
    "national_id",
    "tax_id",
    "url",
    "state",
    "swift_bic",
    "cvv",
    "pin",
]


class EntityModel(Protocol):
    def predict_entities(
        self, text: str, labels: List[str], threshold: float = 0.5
    ) -> List[Dict[str, Any]]: ...


def _chunk_ranges(text_len: int, max_chars: int, overlap: int):
    if text_len <= max_chars:
        yield (0, text_len)
        return
    start = 0
    while start < text_len:
        end = min(start + max_chars, text_len)
        yield (start, end)
        if end == text_len:
            break
        start = end - overlap  # backtrack so we don't split an entity at the boundary

def _chunk_ranges_by_tokens(text: str, tokenizer, max_tokens: int, overlap_tokens: int):
    """
    Token-accurate chunking using the model's own tokenizer, so a page never
    gets silently truncated at the model's 512-token input limit.
    """
    if not text.strip():
        return
    encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    offsets = encoding["offset_mapping"]
    n = len(offsets)
    if n == 0:
        return
    if n <= max_tokens:
        yield (0, len(text))
        return

    start_tok = 0
    while start_tok < n:
        end_tok = min(start_tok + max_tokens, n)
        char_start = offsets[start_tok][0]
        char_end = offsets[end_tok - 1][1]
        yield (char_start, char_end)
        if end_tok == n:
            break
        start_tok = end_tok - overlap_tokens  # back up so an entity isn't split at the boundary

# def detect_entities(
#     model: EntityModel,
#     text: str,
#     labels: List[str] = None,
#     threshold: float = 0.5,
#     max_chars: int = 1800,
#     overlap: int = 200,
# ) -> List[Dict[str, Any]]:
#     """
#     Run the model over `text` in overlapping windows, return a flat,
#     de-duplicated, non-overlapping list of entities in page-absolute
#     [start, end) offsets, each with 'text', 'label', 'score'.
#     """
#     labels = labels or ALL_LABELS
#     raw_hits: List[Dict[str, Any]] = []

#     for chunk_start, chunk_end in _chunk_ranges(len(text), max_chars, overlap):

def detect_entities(
    model: EntityModel,
    text: str,
    labels: List[str] = None,
    threshold: float = 0.5,
    max_tokens: int = 480,      # was: max_chars: int = 1800,
    overlap_tokens: int = 50,   # was: overlap: int = 200,
) -> List[Dict[str, Any]]:
    labels = labels or ALL_LABELS
    raw_hits: List[Dict[str, Any]] = []

    tokenizer = getattr(getattr(model, "data_processor", None), "transformer_tokenizer", None)
    if tokenizer is not None:
        chunk_ranges = _chunk_ranges_by_tokens(text, tokenizer, max_tokens, overlap_tokens)
    else:
        # stub/mock model without a real tokenizer -- rough char estimate fallback
        chunk_ranges = _chunk_ranges(len(text), max_tokens * 4, overlap_tokens * 4)

    for chunk_start, chunk_end in chunk_ranges:
        chunk = text[chunk_start:chunk_end]
        if not chunk.strip():
            continue
        preds = model.predict_entities(chunk, labels, threshold=threshold)
        for p in preds:
            abs_start = chunk_start + p["start"]
            abs_end = chunk_start + p["end"]
            # Guard against a model that doesn't return exact offsets: verify,
            # and if it doesn't line up, locate it directly in the chunk.
            if text[abs_start:abs_end] != p.get("text", text[abs_start:abs_end]):
                local_idx = chunk.find(p["text"])
                if local_idx == -1:
                    continue
                abs_start = chunk_start + local_idx
                abs_end = abs_start + len(p["text"])
            raw_hits.append(
                {
                    "start": abs_start,
                    "end": abs_end,
                    "text": text[abs_start:abs_end],
                    "label": p["label"],
                    "score": p.get("score", 1.0),
                }
            )

    return _resolve_overlaps(raw_hits)

def detect_entities_layered(
    model: EntityModel,
    text: str,
    labels: List[str] = None,
    threshold: float = 0.5,
    max_tokens: int = 480,
    overlap_tokens: int = 50,
    presidio_analyzer=None,
    use_regex: bool = True,
    use_nltk: bool = True,
) -> List[Dict[str, Any]]:
    """
    GLiNER first, then Presidio/regex/NLTK fill in ONLY the gaps GLiNER
    left uncovered -- GLiNER always wins on any overlap.
    """
    primary = detect_entities(model, text, labels, threshold, max_tokens, overlap_tokens)
    covered = [(h["start"], h["end"]) for h in primary]

    secondary_raw = run_secondary_detectors(
        text, presidio_analyzer=presidio_analyzer, use_regex=use_regex, use_nltk=use_nltk
    )
    secondary_clean = [
        h for h in secondary_raw
        if not any(not (h["end"] <= s or h["start"] >= e) for s, e in covered)
    ]
    secondary_clean = _resolve_overlaps(secondary_clean)  # dedupe among themselves too

    merged = primary + secondary_clean
    merged.sort(key=lambda h: h["start"])
    return merged

def _resolve_overlaps(hits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Greedy: prefer higher score, then longer span; drop overlaps; drop exact dupes."""
    seen = set()
    deduped = []
    for h in hits:
        sig = (h["start"], h["end"], h["label"])
        if sig in seen:
            continue
        seen.add(sig)
        deduped.append(h)

    deduped.sort(key=lambda h: (-h["score"], -(h["end"] - h["start"])))

    kept: List[Dict[str, Any]] = []
    occupied: List[tuple] = []
    for h in deduped:
        overlaps = any(not (h["end"] <= s or h["start"] >= e) for s, e in occupied)
        if overlaps:
            continue
        kept.append(h)
        occupied.append((h["start"], h["end"]))

    kept.sort(key=lambda h: h["start"])
    return kept

In [4]:
"""
secondary_detectors.py

Second-pass PII detection that runs AFTER GLiNER, only to catch what it
missed. Combines:
  - Presidio (spaCy-backed NER + its built-in structured-PII recognizers)
  - NLTK (ne_chunk PERSON/ORGANIZATION/GPE) as a lightweight NER fallback
  - Regex for structured formats GLiNER/Presidio/NLTK are all unreliable on
    in Indian financial documents specifically (PAN, Aadhaar, IFSC, Indian
    mobile numbers, GSTIN)

All three return hits in the same dict shape entity_detector.py already
uses: {"start", "end", "text", "label", "score"}. Labels are mapped onto
the SAME label strings pii_mapper.py already knows how to fake, so no
changes are needed there.
"""

!pip install presidio_analyzer
!pip install presidio_anonymizer
!python -m spacy download en_core_web_lg

import re
from typing import Any, Dict, List

# ---------------------------------------------------------------------- #
# Regex pass
# ---------------------------------------------------------------------- #
_REGEX_PATTERNS = [
    ("email", re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")),
    ("url", re.compile(r"\bhttps?://[^\s]+|\bwww\.[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\S*")),
    ("ipv4", re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")),
    ("phone_number", re.compile(r"(?:\+91[\s\-]?)?\b[6-9]\d{9}\b")),
    ("phone_number", re.compile(r"\+\d{1,3}[\s\-]?\d{2,4}[\s\-]?\d{5,8}")),
    ("credit_card_number", re.compile(r"\b(?:\d[ \-]?){13,16}\b")),
    ("national_id", re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b")),          # PAN
    ("national_id", re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b")),        # Aadhaar
    ("bank_routing_number", re.compile(r"\b[A-Z]{4}0[A-Z0-9]{6}\b")),  # IFSC
    ("tax_id", re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z]\d[Z][A-Z\d]\b")),  # GSTIN
    ("postcode", re.compile(r"\b\d{6}\b")),
]


def regex_entities(text: str) -> List[Dict[str, Any]]:
    hits = []
    for label, pattern in _REGEX_PATTERNS:
        for m in pattern.finditer(text):
            hits.append({
                "start": m.start(), "end": m.end(), "text": m.group(0),
                "label": label, "score": 0.55,
            })
    return hits


# ---------------------------------------------------------------------- #
# Presidio pass
# ---------------------------------------------------------------------- #
_PRESIDIO_LABEL_MAP = {
    "PERSON": "name",
    "EMAIL_ADDRESS": "email",
    "PHONE_NUMBER": "phone_number",
    "CREDIT_CARD": "credit_card_number",
    "US_SSN": "ssn",
    "IP_ADDRESS": "ipv4",
    "LOCATION": "address",
    "ORGANIZATION": "company_name",
    "URL": "url",
    "DATE_TIME": "date",
    "IBAN_CODE": "account_number",
    "CRYPTO": "unique_identifier",
    "NRP": "national_id",
}


def presidio_entities(text: str, analyzer) -> List[Dict[str, Any]]:
    if analyzer is None:
        return []
    results = analyzer.analyze(text=text, language="en")
    hits = []
    for r in results:
        label = _PRESIDIO_LABEL_MAP.get(r.entity_type)
        if label is None:
            continue
        hits.append({
            "start": r.start, "end": r.end, "text": text[r.start:r.end],
            "label": label, "score": float(r.score),
        })
    return hits


def build_presidio_analyzer():
    """Call once at startup (loading the spaCy model is slow)."""
    from presidio_analyzer import AnalyzerEngine
    return AnalyzerEngine()


# ---------------------------------------------------------------------- #
# NLTK pass
# ---------------------------------------------------------------------- #
_NLTK_LABEL_MAP = {"PERSON": "name", "ORGANIZATION": "company_name", "GPE": "address"}


def nltk_entities(text: str) -> List[Dict[str, Any]]:
    import nltk

    tokens = nltk.word_tokenize(text)
    tags = nltk.pos_tag(tokens)
    tree = nltk.ne_chunk(tags)

    hits = []
    cursor = 0
    for node in tree:
        if isinstance(node, nltk.Tree):
            label = _NLTK_LABEL_MAP.get(node.label())
            phrase = " ".join(leaf for leaf, _ in node.leaves())
            if label is None:
                continue
            idx = text.find(phrase, cursor)
            if idx == -1:
                continue
            hits.append({
                "start": idx, "end": idx + len(phrase), "text": phrase,
                "label": label, "score": 0.5,
            })
            cursor = idx + len(phrase)
    return hits


# ---------------------------------------------------------------------- #
# Entry point used by entity_detector.py
# ---------------------------------------------------------------------- #
def run_secondary_detectors(text: str, presidio_analyzer=None, use_regex=True, use_nltk=True) -> List[Dict[str, Any]]:
    hits = []
    if use_regex:
        hits += regex_entities(text)
    if presidio_analyzer is not None:
        hits += presidio_entities(text, presidio_analyzer)
    if use_nltk:
        try:
            hits += nltk_entities(text)
        except LookupError:
            pass  # NLTK data not downloaded on this machine -- skip silently
    return hits

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.0/259.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 72.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 64.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
  Attempting uninstall: spacy
    Found existing installation: spacy 3.8.14
    U

In [28]:
"""
ocr_page_handler.py

Handles pages that are scanned images with no extractable text layer (e.g.
a photographed/scanned PAN card as the last page of the prospectus).
Rasterizes the page, runs Tesseract OCR to get word-level text + bounding
boxes, and reshapes that into the exact same (full_text, char_infos) shape
pdf_text_extractor.extract_page_chars() produces -- so everything downstream
(entity detection, entity_to_line_rects, redaction) needs zero changes.
"""

!sudo apt-get install tesseract-ocr

import io
from typing import List, Optional, Tuple

import pymupdf
import pytesseract
from PIL import Image
from pytesseract import Output
import numpy as np
import easyocr

# from pdf_text_extractor import CharInfo

_reader = None

def get_reader(langs=("en",), gpu=False):
    global _reader
    if _reader is None:
        _reader = easyocr.Reader(list(langs), gpu=gpu)
    return _reader

def is_scanned_page(full_text: str, min_chars: int = 20) -> bool:
    """True if the page's normal text layer is (near) empty -- i.e. it's
    probably just an embedded image with no underlying text."""
    return len(full_text.strip()) < min_chars


# def extract_page_chars_ocr(
#     page: "pymupdf.Page", dpi: int = 300, min_conf: int = 10
# ) -> Tuple[str, List[Optional[CharInfo]]]:
#     """
#     OCR equivalent of pdf_text_extractor.extract_page_chars(). Returns
#     (full_text, char_infos) with the same per-character bbox contract, so
#     entity_to_line_rects() works unmodified. Character bboxes are estimated
#     by evenly slicing each OCR word's box across its characters.
#     """
#     zoom = dpi / 72.0
#     pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
#     image = Image.open(io.BytesIO(pix.tobytes("png")))

#     data = pytesseract.image_to_data(image, output_type=Output.DICT)

#     text_chars: List[str] = []
#     char_infos: List[Optional[CharInfo]] = []
#     line_id = 0
#     prev_key = None

#     for i in range(len(data["text"])):
#         word = data["text"][i].strip()
#         try:
#             conf = int(float(data["conf"][i]))
#         except (ValueError, TypeError):
#             conf = -1
#         if not word or conf < min_conf:
#             continue

#         key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
#         if prev_key is not None and key != prev_key:
#             text_chars.append("\n")
#             char_infos.append(None)
#             line_id += 1
#         prev_key = key

#         # pixel coords at `dpi` -> PDF point coords
#         x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
#         px0, py0, px1, py1 = x / zoom, y / zoom, (x + w) / zoom, (y + h) / zoom
#         font_size = (py1 - py0) * 0.8

#         for j, ch in enumerate(word):
#             cx0 = px0 + (px1 - px0) * (j / len(word))
#             cx1 = px0 + (px1 - px0) * ((j + 1) / len(word))
#             text_chars.append(ch)
#             char_infos.append(
#                 CharInfo(ch=ch, bbox=(cx0, py0, cx1, py1), line_id=line_id,
#                          font_size=font_size, font="helv")
#             )
#         text_chars.append(" ")
#         char_infos.append(None)

#     return "".join(text_chars), char_infos

#########################################################

# def extract_page_chars_ocr(
#     page: "pymupdf.Page", dpi: int = 300, min_conf: int = 7, psm: int = 6
# ) -> Tuple[str, List[Optional[CharInfo]]]:
#     """
#     OCR equivalent of pdf_text_extractor.extract_page_chars(). Runs OCR on
#     each embedded image's own region (not the whole rendered page) --
#     Tesseract's page segmentation reliably fails on a mostly-blank page
#     with one small dense photo on it, cropping to the image itself fixes
#     that. Multiple images on one page (e.g. an overlapping ID card scan
#     stacked under another) are each OCR'd and merged.
#     """
#     zoom = dpi / 72.0
#     mat = pymupdf.Matrix(zoom, zoom)
#     full_pix = page.get_pixmap(matrix=mat)
#     full_img = Image.open(io.BytesIO(full_pix.tobytes("png")))

#     image_rects = []
#     for img_info in page.get_images(full=True):
#         xref = img_info[0]
#         image_rects.extend(page.get_image_rects(xref))
#     if not image_rects:
#         image_rects = [page.rect]  # no embedded images found -- fall back to whole page

#     text_chars: List[str] = []
#     char_infos: List[Optional[CharInfo]] = []
#     line_id = 0

#     for rect in image_rects:
#         rect = rect & page.rect  # clip to visible page area
#         if rect.is_empty:
#             continue
#         box = (int(rect.x0 * zoom), int(rect.y0 * zoom), int(rect.x1 * zoom), int(rect.y1 * zoom))
#         crop = full_img.crop(box)
#         data = pytesseract.image_to_data(crop, config=f"--psm {psm}", output_type=Output.DICT)

#         prev_key = None
#         for i in range(len(data["text"])):
#             word = data["text"][i].strip()
#             try:
#                 conf = int(float(data["conf"][i]))
#             except (ValueError, TypeError):
#                 conf = -1
#             if not word or conf < min_conf:
#                 continue

#             key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
#             if prev_key is not None and key != prev_key:
#                 text_chars.append("\n")
#                 char_infos.append(None)
#                 line_id += 1
#             prev_key = key

#             # local crop pixel coords -> page-absolute PDF point coords
#             x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
#             px0, py0 = rect.x0 + x / zoom, rect.y0 + y / zoom
#             px1, py1 = rect.x0 + (x + w) / zoom, rect.y0 + (y + h) / zoom
#             font_size = (py1 - py0) * 0.8

#             for j, ch in enumerate(word):
#                 cx0 = px0 + (px1 - px0) * (j / len(word))
#                 cx1 = px0 + (px1 - px0) * ((j + 1) / len(word))
#                 text_chars.append(ch)
#                 char_infos.append(
#                     CharInfo(ch=ch, bbox=(cx0, py0, cx1, py1), line_id=line_id,
#                              font_size=font_size, font="helv")
#                 )
#             text_chars.append(" ")
#             char_infos.append(None)

#         # separate each image's OCR text with a line break
#         text_chars.append("\n")
#         char_infos.append(None)
#         line_id += 1

#     return "".join(text_chars), char_infos

##############################################################

def extract_page_chars_ocr(
    page: "pymupdf.Page", dpi: int = 300, min_conf: int = 7, gpu: bool = False
) -> Tuple[str, List[Optional[CharInfo]]]:
    zoom = dpi / 72.0
    mat = pymupdf.Matrix(zoom, zoom)
    full_pix = page.get_pixmap(matrix=mat)
    full_img = Image.open(io.BytesIO(full_pix.tobytes("png")))

    image_rects = []
    for img_info in page.get_images(full=True):
        xref = img_info[0]
        image_rects.extend(page.get_image_rects(xref))
    if not image_rects:
        image_rects = [page.rect]

    reader = get_reader(gpu=gpu)

    text_chars: List[str] = []
    char_infos: List[Optional[CharInfo]] = []
    line_id = 0

    for rect in image_rects:
        rect = rect & page.rect
        if rect.is_empty:
            continue
        box = (int(rect.x0 * zoom), int(rect.y0 * zoom), int(rect.x1 * zoom), int(rect.y1 * zoom))
        crop = full_img.crop(box)
        crop_np = np.array(crop.convert("RGB"))

        detections = reader.readtext(crop_np)  # [(quad_bbox, text, conf), ...]

        for quad, text, conf in detections:
            text = text.strip()
            if not text or conf * 100 < min_conf:
                continue

            xs = [p[0] for p in quad]
            ys = [p[1] for p in quad]
            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
            px0, py0 = rect.x0 + x0 / zoom, rect.y0 + y0 / zoom
            px1, py1 = rect.x0 + x1 / zoom, rect.y0 + y1 / zoom
            font_size = (py1 - py0) * 0.8

            # EasyOCR returns a whole phrase per detection (not word-by-word
            # like Tesseract), so slice the box evenly across every char
            # of the phrase, spaces included.
            n = len(text)
            for j, ch in enumerate(text):
                cx0 = px0 + (px1 - px0) * (j / n)
                cx1 = px0 + (px1 - px0) * ((j + 1) / n)
                text_chars.append(ch)
                char_infos.append(
                    CharInfo(ch=ch, bbox=(cx0, py0, cx1, py1), line_id=line_id,
                             font_size=font_size, font="helv")
                )
            text_chars.append("\n")
            char_infos.append(None)
            line_id += 1

        text_chars.append("\n")
        char_infos.append(None)
        line_id += 1

    return "".join(text_chars), char_infos

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 124 not upgraded.


In [25]:
"""
pii_mapper.py

Turns a (label, original_text) pair into a fake replacement, and remembers
the mapping so the SAME original value always gets the SAME fake value
everywhere in the document (e.g. every occurrence of "Sarthak Malvadkar"
across all 128 pages becomes the same fake name).

Design:
  - One cache keyed on the *normalized* original string (casefolded,
    whitespace-collapsed) so trivial formatting differences ("Rohan Dey"
    vs "Rohan  Dey") still resolve to one fake value.
  - Per-label generator functions so the fake value "looks like" the real
    one (an email stays an email, a phone number stays phone-number
    shaped, etc). Anything without a bespoke generator falls back to a
    format-preserving scrambler (digits->digits, letters->letters) so at
    minimum length/punctuation/shape is preserved.
"""

!pip install Faker

import random
import re
import string
from typing import Callable, Dict

from faker import Faker


class PIIMapper:
    def __init__(self, seed: int = 42):
        self.fake = Faker()
        Faker.seed(seed)
        random.seed(seed)
        self._cache: Dict[str, str] = {}
        self._generators: Dict[str, Callable[[str], str]] = self._build_generators()

    # ---------------------------------------------------------------- #
    # public API
    # ---------------------------------------------------------------- #
    def get_fake(self, label: str, original: str) -> str:
        key = self._norm_key(label, original)
        if key in self._cache:
            return self._cache[key]

        gen = self._generators.get(label, self._format_preserving)
        try:
            fake_value = gen(original)
        except Exception:
            fake_value = self._format_preserving(original)

        self._cache[key] = fake_value
        return fake_value

    def export_mapping(self) -> Dict[str, str]:
        """original -> fake, for the audit log / README appendix."""
        return dict(self._cache)

    # ---------------------------------------------------------------- #
    # internals
    # ---------------------------------------------------------------- #
    def _norm_key(self, label: str, original: str) -> str:
        # Names/emails/etc. should map consistently regardless of which
        # GLiNER label first, last, or full-name fired on. We key primarily
        # on the normalized text; label is only a tiebreaker for the rare
        # case the same string legitimately means different things.
        norm_text = re.sub(r"\s+", " ", original.strip().casefold())
        return norm_text

    def _split_full_name(self, original: str):
        parts = original.strip().split()
        first = parts[0] if parts else "John"
        last = parts[-1] if len(parts) > 1 else "Doe"
        return first, last

    def _fake_name_consistent(self, original: str) -> str:
        first, last = self._split_full_name(original)
        f_key = self._norm_key("first_name", first)
        l_key = self._norm_key("last_name", last)
        fake_first = self._cache.get(f_key) or self.fake.first_name()
        fake_last = self._cache.get(l_key) or self.fake.last_name()
        self._cache[f_key] = fake_first
        self._cache[l_key] = fake_last
        if len(original.strip().split()) == 1:
            return fake_first
        return f"{fake_first} {fake_last}"

    def _fake_email(self, original: str) -> str:
        # Try to build "first.last@example.com" the way real names do,
        # for readability -- falls back to a random email otherwise.
        m = re.match(r"([a-zA-Z]+)[._-]?([a-zA-Z]*)@", original)
        if m and m.group(2):
            first, last = m.group(1), m.group(2)
            fake_first = self.fake.first_name().lower()
            fake_last = self.fake.last_name().lower()
            return f"{fake_first}.{fake_last}@example.com"
        return self.fake.first_name().lower() + "." + self.fake.last_name().lower() + "@example.com"

    def _fake_phone(self, original: str) -> str:
        digits = re.findall(r"\d", original)
        prefix = "+91 " if original.strip().startswith("+91") else ""
        if len(digits) >= 10:
            new_digits = [random.choice(string.digits) for _ in range(len(digits))]
            # keep an Indian-looking leading digit (6-9) if original did
            if digits[0] in "6789":
                new_digits[0] = random.choice("6789")
            return prefix + "".join(new_digits[-10:]) if prefix else "".join(new_digits)
        return self.fake.phone_number()

    def _fake_date(self, original: str) -> str:
        fmt_guess = self._guess_date_format(original)
        d = self.fake.date_of_birth(minimum_age=1, maximum_age=90)
        try:
            return d.strftime(fmt_guess)
        except Exception:
            return d.isoformat()

    def _guess_date_format(self, original: str) -> str:
        if re.match(r"\d{2}/\d{2}/\d{4}", original):
            return "%d/%m/%Y"
        if re.match(r"\d{4}-\d{2}-\d{2}", original):
            return "%Y-%m-%d"
        if re.search(r"[A-Za-z]", original):
            return "%d %B %Y"
        return "%d/%m/%Y"

    # def _format_preserving(self, original: str) -> str:
    #     """
    #     Generic fallback: same length & punctuation, digits replaced with
    #     digits, letters replaced with letters of the same case. Used for
    #     the long tail of structured-ID labels (account numbers, license
    #     plates, API keys, etc.) where we don't have a bespoke Faker
    #     provider but still want something plausible-looking rather than
    #     just "REDACTED".
    #     """
    #     out = []
    #     for ch in original:
    #         if ch.isdigit():
    #             out.append(random.choice(string.digits))
    #         elif ch.isupper():
    #             out.append(random.choice(string.ascii_uppercase))
    #         elif ch.islower():
    #             out.append(random.choice(string.ascii_lowercase))
    #         else:
    #             out.append(ch)
    #     return "".join(out)

    def _format_preserving(self, original: str) -> str:
        """
        Mask: every digit/letter becomes 'X', punctuation and spacing (dashes,
        parens, slashes) are preserved so the shape stays recognizable without
        fabricating a plausible-looking number.
        """
        out = []
        for ch in original:
            if ch.isdigit() or ch.isalpha():
                out.append("X")
            else:
                out.append(ch)
        return "".join(out)

    def _build_generators(self) -> Dict[str, Callable[[str], str]]:
        f = self.fake
        return {
            "name": self._fake_name_consistent,
            "first_name": lambda o: f.first_name(),
            "last_name": lambda o: f.last_name(),
            "email": self._fake_email,
            "phone_number": lambda o: self._format_preserving(o),
            "company_name": lambda o: f.company(),
            "street_address": lambda o: f.street_address(),
            "address": lambda o: f.address().replace("\n", ", "),
            "city": lambda o: f.city(),
            "state": lambda o: f.state(),
            "country": lambda o: f.country(),
            "postcode": lambda o: self._format_preserving(o),
            "coordinate": lambda o: f"{self._format_preserving(o)}, {self._format_preserving(o)}",
            "date_of_birth": self._fake_date,
            "date": self._fake_date,
            "date_time": self._fake_date,
            "time": lambda o: f.time(),
            "ssn": lambda o: self._format_preserving(o),
            "national_id": lambda o: self._format_preserving(o),
            "tax_id": lambda o: self._format_preserving(o),
            "credit_card_number": lambda o: self._format_preserving(o),
            "cvv": lambda o: self._format_preserving(o),
            "bank_routing_number": lambda o: self._format_preserving(o),
            "account_number": lambda o: self._format_preserving(o),
            "swift_bic": lambda o: self._format_preserving(o),
            "ipv4": lambda o: f.ipv4(),
            "ipv6": lambda o: f.ipv6(),
            "url": lambda o: f.url(),
            "user_name": lambda o: f.user_name(),
            "password": lambda o: self._format_preserving(o),
            "api_key": lambda o: self._format_preserving(o),
            "pin": lambda o: self._format_preserving(o),
            "employee_id": lambda o: self._format_preserving(o),
            "customer_id": lambda o: self._format_preserving(o),
            "medical_record_number": lambda o: self._format_preserving(o),
            "health_plan_beneficiary_number": lambda o: self._format_preserving(o),
            "certificate_license_number": lambda o: self._format_preserving(o),
            "vehicle_identifier": lambda o: self._format_preserving(o),
            "license_plate": lambda o: self._format_preserving(o),
            "device_identifier": lambda o: self._format_preserving(o),
            "biometric_identifier": lambda o: self._format_preserving(o),
            "unique_identifier": lambda o: self._format_preserving(o),
        }

In [26]:
!pip install presidio-analyzer presidio-anonymizer

In [33]:
#!/usr/bin/env python3
"""
redact_pdf.py

Redacts PII from a PDF, page by page, replacing each detected value with a
consistent fake alternative -- while preserving the original page layout
(fonts, tables, images, logos, QR codes, etc. are left untouched).

Pipeline per page:
  1. page.get_text("rawdict") -> plain text + char-level bbox map
     (pdf_text_extractor.py)
  2. GLiNER (gretelai/gretel-gliner-bi-base-v1.0) finds PII spans in that
     text, chunked to respect the model's context limit
     (entity_detector.py)
  3. Each span is mapped to a consistent fake value
     (pii_mapper.py)
  4. Each span's char offsets are mapped back to on-page rectangles
     (pdf_text_extractor.entity_to_line_rects)
  5. page.add_redact_annot() blanks the original rectangles,
     page.apply_redactions() burns that in (original text is gone, not
     just hidden), then page.insert_textbox() writes the fake value into
     the same spot.

Usage:
    python redact_pdf.py input.pdf output.pdf \
        --model gretelai/gretel-gliner-bi-base-v1.0 \
        --threshold 0.5 \
        --log mapping_log.json
"""

!pip install gliner

import argparse
import json
import sys
import time

import pymupdf  # PyMuPDF

# from entity_detector import ALL_LABELS, detect_entities_layered
# from pdf_text_extractor import entity_to_line_rects, extract_page_chars
# from pii_mapper import PIIMapper
# import ocr_page_handler


def load_model(model_name: str):
    from gliner import GLiNER

    print(f"[*] Loading {model_name} ...", file=sys.stderr)
    model = GLiNER.from_pretrained(model_name)
    return model


def redact_page(page: "pymupdf.Page", model, mapper: PIIMapper, threshold: float, labels, keep_substrings=None, presidio_analyzer=None):
    full_text, char_infos = extract_page_chars(page)
    # if not full_text.strip():
    #     return []

    # # entities = detect_entities(model, full_text, labels=labels, threshold=threshold)
    # entities = detect_entities_layered(model, full_text, labels=labels, threshold=threshold, presidio_analyzer=presidio_analyzer, use_nltk=False)
    is_scanned = is_scanned_page(full_text)
    # has_images = len(page.get_images(full=True)) > 0
    if is_scanned:
        full_text, char_infos = extract_page_chars_ocr(page)

    if not full_text.strip():
        return []

    if is_scanned:
        # Scanned page (e.g. the PAN card image): only GLiNER + Presidio,
        # per instructions -- regex/NLTK skipped since OCR text is noisier
        # and those two passes are tuned for clean extracted PDF text.
        entities = detect_entities_layered(
            model, full_text, labels=labels, threshold=threshold,
            presidio_analyzer=presidio_analyzer, use_regex=False, use_nltk=False,
        )
    else:
        entities = detect_entities_layered(
            model, full_text, labels=labels, threshold=threshold,
            presidio_analyzer=presidio_analyzer, use_nltk=False,
        )
    keep_substrings = [s.lower() for s in (keep_substrings or [])]

    plan = []
    for ent in entities:
        if ent["label"] == "company_name" and any(k in ent["text"].lower() for k in keep_substrings):
            # This is the document's own subject company, not third-party
            # PII -- see README "Design choices" for why this is excluded.
            continue
        rects = entity_to_line_rects(ent["start"], ent["end"], char_infos)
        if not rects:
            continue
        fake = mapper.get_fake(ent["label"], ent["text"])
        plan.append({**ent, "rects": rects, "fake": fake})

    if not plan:
        return []

    # Phase 1: blank out every rect for every entity on this page.
    for item in plan:
        for r in item["rects"]:
            page.add_redact_annot(r["rect"], fill=(0, 0, 0))
    page.apply_redactions(images=pymupdf.PDF_REDACT_IMAGE_PIXELS if is_scanned else pymupdf.PDF_REDACT_IMAGE_NONE)

    # Phase 2: write the fake replacement into the freshly-blanked space.
    for item in plan:
        _insert_replacement(page, item)

    return plan



# insert_textbox() needs noticeably more vertical room than the glyphs'
# own bbox height to lay a line out without reporting overflow (it budgets
# for ascent/descent/leading, not just ink height). Empirically, giving it
# a box ~1.8x the fontsize tall is what makes single-line inserts succeed
# reliably, so we size the font off of the *available* rect height using
# the inverse of that ratio rather than the other way around.
_LINE_HEIGHT_RATIO = 1.75
_BOX_HEIGHT_RATIO = 1.8


def _insert_replacement(page: "pymupdf.Page", item: dict):
    rects = item["rects"]
    fake = item["fake"]

    if len(rects) == 1:
        r = rects[0]["rect"]
        # fs = _fit_fontsize(r.height, rects[0]["font_size"])
        fs, text = _fit_fontsize_and_text(fake, r, rects[0]["font_size"])
        needed_w = pymupdf.get_text_length(fake, fontname="helv", fontsize=fs)
        cy = (r.y0 + r.y1) / 2
        box_h = fs * _BOX_HEIGHT_RATIO
        # box = pymupdf.Rect(
        #     r.x0 - 1, cy - box_h / 2,
        #     max(r.x1, r.x0 + needed_w + 4), cy + box_h / 2,
        # )
        box = pymupdf.Rect(r.x0, cy - box_h / 2, r.x1, cy + box_h / 2)
        # rc = page.insert_textbox(
        #     box, fake, fontsize=fs, fontname="helv", color=(1, 0, 0), align=0
        # )
        rc = page.insert_textbox(
            box, text, fontsize=fs, fontname="helv", color=(1, 0, 0), align=0
        )
        # if rc < 0:
        #     for smaller in (fs * 0.85, fs * 0.7, 5.0):
        #         box = pymupdf.Rect(
        #             r.x0 - 1, cy - (smaller * _BOX_HEIGHT_RATIO) / 2,
        #             max(r.x1, r.x0 + needed_w + 4), cy + (smaller * _BOX_HEIGHT_RATIO) / 2,
        #         )
        #         rc = page.insert_textbox(
        #             box, fake, fontsize=smaller, fontname="helv", color=(1, 0, 0), align=0
        #         )
        #         if rc >= 0:
        #             break
        if rc < 0:
            for smaller in (fs * 0.85, fs * 0.7, min(fs, 5.0)):
                fs2, text2 = _fit_fontsize_and_text(fake, r, smaller)
                box = pymupdf.Rect(r.x0, cy - (fs2 * _BOX_HEIGHT_RATIO) / 2, r.x1, cy + (fs2 * _BOX_HEIGHT_RATIO) / 2)
                rc = page.insert_textbox(
                    box, text2, fontsize=fs2, fontname="helv", color=(1, 0, 0), align=0
                )
                if rc >= 0:
                    break
    # else:
    #     x0 = min(r["rect"].x0 for r in rects)
    #     y0 = min(r["rect"].y0 for r in rects)
    #     x1 = max(r["rect"].x1 for r in rects)
    #     y1 = max(r["rect"].y1 for r in rects)
    #     avg_line_h = (y1 - y0) / len(rects)
    #     fs = _fit_fontsize(avg_line_h, sum(r["font_size"] for r in rects) / len(rects))
    #     pad = fs * (_BOX_HEIGHT_RATIO - 1) * len(rects)
    #     box = pymupdf.Rect(x0 - 1, y0 - pad / 2, x1 + 2, y1 + pad / 2)
    #     page.insert_textbox(
    #         box, fake, fontsize=fs, fontname="helv", color=(1, 0, 0), align=0
    #     )
    else:
        x0 = min(r["rect"].x0 for r in rects)
        y0 = min(r["rect"].y0 for r in rects)
        x1 = max(r["rect"].x1 for r in rects)
        y1 = max(r["rect"].y1 for r in rects)
        union_rect = pymupdf.Rect(x0, y0, x1, y1)
        avg_line_h = (y1 - y0) / len(rects)
        fs, text = _fit_fontsize_and_text(fake, union_rect, sum(r["font_size"] for r in rects) / len(rects))
        pad = fs * (_BOX_HEIGHT_RATIO - 1) * len(rects)
        box = pymupdf.Rect(x0, y0 - pad / 2, x1, y1 + pad / 2)
        page.insert_textbox(
            box, text, fontsize=fs, fontname="helv", color=(1, 0, 0), align=0
        )


def _fit_fontsize(available_height: float, original_size: float) -> float:
    fs = min(original_size, available_height / _LINE_HEIGHT_RATIO)
    return round(max(4.0, fs), 1)

def _fit_fontsize_and_text(fake: str, rect: "pymupdf.Rect", original_size: float, min_size: float = 4.0):
    """
    Returns (fontsize, text) such that `text` rendered at `fontsize` fits
    inside `rect` both vertically and horizontally -- i.e. stays within the
    black box, never outgrows it. Shrinks the font first; if it's still too
    wide even at min_size, truncates with an ellipsis as a last resort.
    """
    fs = min(original_size, rect.height / _LINE_HEIGHT_RATIO)
    fs = max(fs, min_size)

    unit_w = pymupdf.get_text_length(fake, fontname="helv", fontsize=1.0)
    if unit_w > 0:
        fs = min(fs, rect.width / unit_w)
    fs = round(max(fs, min_size), 1)

    text = fake
    if pymupdf.get_text_length(text, fontname="helv", fontsize=fs) > rect.width:
        while text and pymupdf.get_text_length(text + "…", fontname="helv", fontsize=fs) > rect.width:
            text = text[:-1]
        text = (text + "…") if text else "…"

    return fs, text


def main():

    # import secondary_detectors

    input_pdf = "/kaggle/input/datasets/arghakamalsamanta/input-data/Red Herring Prospectus.pdf"
    output_pdf = "redacted_output.pdf"
    model = "gretelai/gretel-gliner-bi-base-v1.0"
    threshold = 0.5
    log = "mapping_log.json"
    start_page = 0
    keep_company = "KSH International"

    model = load_model(model)
    mapper = PIIMapper(seed=42)
    presidio_analyzer = build_presidio_analyzer() 

    doc = pymupdf.open(input_pdf)
    end_page = len(doc)

    t0 = time.time()
    total_entities = 0
    for i in range(start_page, end_page):
        page = doc[i]
        plan = redact_page(page, model, mapper, threshold, ALL_LABELS, keep_substrings=keep_company, presidio_analyzer=presidio_analyzer)
        total_entities += len(plan)
        print(f"[page {i + 1}/{len(doc)}] {len(plan)} PII spans redacted", file=sys.stderr)

    doc.save(output_pdf, garbage=4, deflate=True)
    doc.close()

    if log:
        with open(log, "w") as f:
            json.dump(mapper.export_mapping(), f, indent=2)

    dt = time.time() - t0
    print(f"[*] Done. {total_entities} total spans redacted in {dt:.1f}s -> {output_pdf}", file=sys.stderr)


if __name__ == "__main__":
    main()

[*] Loading gretelai/gretel-gliner-bi-base-v1.0 ...


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[page 1/128] 21 PII spans redacted
[page 2/128] 2 PII spans redacted
[page 3/128] 5 PII spans redacted
[page 4/128] 17 PII spans redacted
[page 5/128] 52 PII spans redacted
[page 6/128] 39 PII spans redacted
[page 7/128] 0 PII spans redacted
[page 8/128] 30 PII spans redacted
[page 9/128] 8 PII spans redacted
[page 10/128] 30 PII spans redacted
[page 11/128] 21 PII spans redacted
[page 12/128] 40 PII spans redacted
[page 13/128] 1 PII spans redacted
[page 14/128] 1 PII spans redacted
[page 15/128] 7 PII spans redacted
[page 16/128] 13 PII spans redacted
[page 17/128] 2 PII spans redacted
[page 18/128] 10 PII spans redacted
[page 19/128] 2 PII spans redacted
[page 20/128] 15 PII spans redacted
[page 21/128] 4 PII spans redacted
[page 22/128] 11 PII spans redacted
[page 23/128] 2 PII spans redacted
[page 24/128] 21 PII spans redacted
[page 25/128] 6 PII spans redacted
[page 26/128] 26 PII spans redacted
[page 27/128] 10 PII spans redacted
[page 28/128] 24 PII spans redacted
[page 29/128]